# ViT ミニマム動作確認（XLA無効 + batch=32）

In [1]:
import os
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'

import tensorflow as tf
from tensorflow.keras import layers
import numpy as np
import time

tf.config.optimizer.set_jit(False)
print('JIT:', tf.config.optimizer.get_jit())

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    tf.config.set_logical_device_configuration(gpus[0], [tf.config.LogicalDeviceConfiguration(memory_limit=18155)])
    print('GPU:', gpus[0].name)
else:
    print('GPUなし → 中断')

W0000 00:00:1782169412.387494    3556 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782169412.387617    3556 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782169412.387626    3556 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782169412.387631    3556 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
2026-06-22 23:03:32.396963: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE3 SSE4.1 SSE4.2 AVX AVX2 FMA, in other operations, rebuild TensorFlow wi

JIT: 
GPU: /physical_device:GPU:0


In [2]:
tf.keras.mixed_precision.set_global_policy('mixed_float16')
print('混合精度:', tf.keras.mixed_precision.global_policy().name)

# 最小ViT-B/16
class AddClsToken(layers.Layer):
    def __init__(self, pdim, **kwargs):
        super().__init__(**kwargs); self.pdim = pdim
    def build(self, input_shape):
        self.cls_token = self.add_weight(shape=(1,1,self.pdim), initializer='random_normal', trainable=True, name='cls_token')
    def call(self, x):
        from tensorflow.keras import ops as k_ops
        return k_ops.broadcast_to(self.cls_token, [k_ops.shape(x)[0], 1, self.pdim])

t0 = time.time()
inputs = layers.Input(shape=(224,224,3))
x = layers.Conv2D(768, 16, 16, padding='valid', name='conv_proj')(inputs)
x = layers.Reshape((-1, 768), name='reshape_patches')(x)
c = AddClsToken(768, name='cls_token')(x)
x = layers.Concatenate(axis=1, name='concat_cls')([c, x])
pos = tf.range(0, 197, 1); pe = layers.Embedding(197, 768, name='pos_embed')
x = layers.Add(name='add_pos_embed')([x, tf.expand_dims(pe(pos), 0)])
x = layers.Dropout(0.0, name='dropout')(x)

for i in range(12):
    x1 = layers.LayerNormalization(epsilon=1e-6, name=f'ln1_{i}')(x)
    attn = layers.MultiHeadAttention(num_heads=12, key_dim=64, name=f'mha_{i}')(x1, x1)
    x = layers.Add(name=f'res1_{i}')([x, attn])
    x2 = layers.LayerNormalization(epsilon=1e-6, name=f'ln2_{i}')(x)
    mlp = layers.Dense(3072, activation='gelu', name=f'mlp1_{i}')(x2)
    mlp = layers.Dense(768, name=f'mlp2_{i}')(mlp)
    x = layers.Add(name=f'res2_{i}')([x, mlp])

x = layers.LayerNormalization(epsilon=1e-6, name='ln_final')(x)
out = layers.Dense(2, activation='softmax', dtype='float32', name='head')(x[:,0])
model = tf.keras.Model(inputs, out, name='ViT_B16')
print(f'モデル構築OK: {model.count_params():,} params ({time.time()-t0:.1f}秒)')

混合精度: mixed_float16


2026-06-22 23:03:43.123223: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1782169423.123348    3556 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 18155 MB memory:  -> device: 0, name: AMD Radeon RX 7900 XT, pci bus id: 0000:2d:00.0


モデル構築OK: 85,648,898 params (4.0秒)


In [3]:
# batch=32 で1バッチ訓練
BATCH = 64
x_dummy = np.random.randn(BATCH, 224, 224, 3).astype('float32') * 2.0 - 1.0
y_dummy = np.random.randint(0, 2, size=(BATCH,)).astype('int32')

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'], jit_compile=False)
print(f'コンパイル完了（XLA完全無効）、batch={BATCH} で訓練開始…')
t0 = time.time()
history = model.fit(x_dummy, y_dummy, batch_size=BATCH, epochs=1, verbose=1)
print(f'✅ 訓練成功: {time.time()-t0:.1f}秒')
print(f'Loss: {history.history["loss"][0]:.4f}, Acc: {history.history["accuracy"][0]:.4f}')

コンパイル完了（XLA完全無効）、batch=64 で訓練開始…
1/1 ━━━━━━━━━━━━━━━━━━━━ 42s 42s/step - accuracy: 0.4531 - loss: 0.9861
✅ 訓練成功: 42.3秒
Loss: 0.9861, Acc: 0.4531
